# PhysAR-IDS -- NB5c (v6): did the white-box attack converge against the robust models?

**Why.** In NB5b the greedy white-box attack stopped after 60 rounds. Against the robust models, a share of attack rows had neither evaded nor run out of improving moves when it stopped (about 4% of rows at 0.3 sigma, 7% at 0.5 sigma, 17% at 1.0 sigma). The evasion rates reported there are therefore lower bounds, and the ransomware / reverseshell result at 1.0 sigma in particular is not yet established. This notebook reloads the five saved robust models (`xgb_robust_v6_seed{s}.json`, no retraining) and gives the attacker up to **300 rounds** at 0.3, 0.5 and 1.0 sigma on the **same 10,000 test rows**, as NB3d did for the standard model.

**Reported:** evasion at 60 / 100 / 150 / 200 / 300 rounds (60 must reproduce NB5b), per class; the share of rows still improving at the end; whether the curve is flat; and the final robust-vs-NB5 comparison with paired tests. The decision on each row is still made by the CPU model. The result is still a *greedy* attack, so it remains a lower bound on what an optimal attacker could do, but the budget-limited stopping is removed.

Needs: `xgb_robust_v6_seed{s}.json`, `defence1b_seed{s}_v6.pkl`, `defence1_seed{s}_v6.pkl`, and the same data files as NB5. Runtime about 40 min per seed.

## 0. Setup

In [ ]:
import os, sys, random, time, json, hashlib, shutil, platform, warnings, copy
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import xgboost as xgb
from scipy import stats
from sklearn.metrics import f1_score
import matplotlib; import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
try:
    from sb3_contrib import RecurrentPPO
    from stable_baselines3.common.callbacks import CheckpointCallback
except ImportError as e:
    raise ImportError('Install in a terminal (not in a cell): pip install stable-baselines3 sb3-contrib gymnasium --no-deps') from e
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False

# ---- PPO / environment settings (unchanged from the reference design) ----
EPS_BOUND = 0.3; MAX_STEPS = 10
EVASION_REWARD = 10.0; STEP_PENALTY_SCALE = 0.01; TIMEOUT_PENALTY = -1.0
TOTAL_TIMESTEPS = 50000; CKPT_FREQ = 5000
EPS_TOTAL_BUDGET = 0.1
CLIP_BUDGETS = {'1.0': 1.0, 'unconstrained': None}     # budgets at which agents are RETRAINED with range clipping (stage B)
SAVED_AGENTS = {'0.1': ('ppo_attack2_budgeted_{v}_seed{s}', 0.1), '0.2': ('ppo_attack2_budget0p2_{v}_seed{s}', 0.2), '0.3': ('ppo_attack2_budget0p3_{v}_seed{s}', 0.3),
                '0.5': ('ppo_attack2_budget0p5_{v}_seed{s}', 0.5), '1.0': ('ppo_attack2_budget1p0_{v}_seed{s}', 1.0), 'unconstrained': ('ppo_attack2_unconstrained_{v}_seed{s}', None)}
EVAL_CHUNK = 8192


# ---- Defence 1 (adversarial training) settings ----
PGD_EPS = 0.1; PGD_ITERS = 10                       # as NB3 / NB3b (all 93 features, alpha = eps / 4)
TRAIN_PPO_TAGS = ['0.1', '0.3']                     # PPO agents (NB4 / NB4b, trained against the STANDARD model) whose examples go into the training set
EVAL_PPO_TAGS = ['0.1', '0.3', '0.5', '1.0', 'unconstrained']   # budgets 0.5 / 1.0 / unconstrained were never seen in training
N_PPO_TRAIN_PER_CLASS = 8000
AUG_RATIO = 1.0                                     # adversarial training rows per clean row (the paper's 1:1)
# ---- white-box (tree-greedy, NB3c / NB3d) attack against both models ----
K_CAND = 12; MAX_ITERS = 60; CHUNK = 20000; BLOCK = 300_000
WB_BUDGETS = {'0.1': 0.1, '0.3': 0.3, '0.5': 0.5, '1.0': 1.0}
N_WB_PER_CLASS = 2000                               # attack rows per class searched (same rows for every seed, model and budget)


# ---- NB5c: convergence of the white-box attack against the NB5b robust models ----
LONG_ITERS = 300                              # NB5b / NB5 used MAX_ITERS = 60
CHECKPOINTS = [60, 100, 150, 200, 300]
LONG_TAGS = ['0.3', '0.5', '1.0']             # 0.1 sigma already converged in NB5b (no rows at the round limit)

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
BACKEND_NAME = 'cpu(hist)' if _cpu else 'gpu(cuda,hist)'
print(f'torch {torch.__version__} ({DEVICE}) | xgboost {xgb.__version__} fit backend: {BACKEND_NAME}')
import sb3_contrib, stable_baselines3
print(f'sb3-contrib {sb3_contrib.__version__} | stable-baselines3 {stable_baselines3.__version__} | gymnasium {gym.__version__}')
print('Working directory:', os.getcwd())

## 1. Load v6 data

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]; N_CYBER = len(CYBER_IDX)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)

atk_tr = y_train != 0; X_atk_train, y_atk_train = X_train[atk_tr], y_train[atk_tr]
atk_te = y_test != 0;  X_atk_test,  y_atk_test  = X_test[atk_te],  y_test[atk_te]
test_burst = pd.factorize(pd.MultiIndex.from_arrays([y_test, np.asarray(data['PLCID_test']), np.asarray(data['timestamp_test'])]))[0][atk_te]
FEAT_LO, FEAT_HI = X_train.min(0), X_train.max(0)
print(f'Features {N_FEAT} | attacker-controlled (cyber) {N_CYBER} | training pool {X_atk_train.shape} | evaluation rows {X_atk_test.shape} in {len(np.unique(test_burst))} attack bursts')
print('Training pool class counts:', np.bincount(y_atk_train)[1:], '| evaluation class counts:', np.bincount(y_atk_test)[1:])
nb2 = joblib.load(f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl')
nb2_tuned = np.array([nb2['results']['tuned'][s]['test_f1'] for s in SEEDS if s in nb2['results']['tuned']])
print(f'NB2 tuned test F1 over seeds: {nb2_tuned.mean():.4f} +/- {nb2_tuned.std(ddof=1):.4f}')

## 2. Oracle

In [ ]:
class Oracle:
    """Hard-label black-box oracle over the fitted XGBoost classifier."""
    def __init__(self, clf): self.clf = clf; self.n_queries = 0
    def predict(self, X):
        X = np.ascontiguousarray(X, dtype=np.float32); self.n_queries += len(X)
        return self.clf.predict(X)

def apply_action(x_cur, x_orig, action, eps_bound=EPS_BOUND, eps_total=None, range_clip=False):
    """In place on x_cur[:, CYBER_IDX]; returns the applied (post-clamp) perturbation."""
    a = np.clip(action, -eps_bound, eps_bound).astype(np.float32)
    old = x_cur[:, CYBER_IDX].copy(); prop = old + a
    if eps_total is not None:
        o = x_orig[:, CYBER_IDX]; prop = o + np.clip(prop - o, -eps_total, eps_total)
    if range_clip:   # never leave the observed training range [min, max] (a value already outside it may stay where it was, but may not go further out)
        prop = np.clip(prop, np.minimum(FEAT_LO[CYBER_IDX], x_orig[:, CYBER_IDX]), np.maximum(FEAT_HI[CYBER_IDX], x_orig[:, CYBER_IDX]))
    x_cur[:, CYBER_IDX] = prop
    return (prop - old).astype(np.float32)

def build_obs(x_cur, last_applied, last_pred, step):
    oh = np.zeros((len(x_cur), 6), dtype=np.float32); oh[np.arange(len(x_cur)), last_pred] = 1.0
    return np.concatenate([x_cur.astype(np.float32), last_applied.astype(np.float32), oh, np.full((len(x_cur), 1), step / MAX_STEPS, dtype=np.float32)], 1)

class IDSEvasionEnv(gym.Env):
    metadata = {'render_modes': []}
    def __init__(self, X_pool, y_pool, oracle, eps_bound=EPS_BOUND, max_steps=MAX_STEPS, seed=None, eps_total_budget=None, range_clip=False):
        super().__init__(); self.range_clip = range_clip
        self.X_pool, self.y_pool, self.oracle = X_pool, y_pool, oracle
        self.eps_bound, self.max_steps, self.eps_total_budget = eps_bound, max_steps, eps_total_budget
        self.rng = np.random.RandomState(seed)
        self.action_space = spaces.Box(low=-eps_bound, high=eps_bound, shape=(N_CYBER,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(N_FEAT + N_CYBER + 6 + 1,), dtype=np.float32)
    def _obs(self): return build_obs(self.x_current[None], self.last_applied[None], np.array([self.last_pred]), self.step_count)[0]
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        idx = self.rng.randint(0, len(self.X_pool))
        self.x_orig = self.X_pool[idx].copy(); self.x_current = self.x_orig.copy(); self.y_true = self.y_pool[idx]
        self.step_count = 0; self.last_applied = np.zeros(N_CYBER, dtype=np.float32)
        self.last_pred = int(self.oracle.predict(self.x_current[None])[0]); return self._obs(), {}
    def step(self, action):
        applied = apply_action(self.x_current[None], self.x_orig[None], np.asarray(action, dtype=np.float32)[None], self.eps_bound, self.eps_total_budget, self.range_clip)[0]
        self.last_applied = applied; self.step_count += 1
        self.last_pred = int(self.oracle.predict(self.x_current[None])[0]); pen = STEP_PENALTY_SCALE * float(np.sum(applied ** 2))
        terminated = truncated = False
        if self.last_pred == 0: reward = EVASION_REWARD - pen; terminated = True
        elif self.step_count >= self.max_steps: reward = TIMEOUT_PENALTY - pen; truncated = True
        else: reward = -pen
        return self._obs(), reward, terminated, truncated, {'evaded': self.last_pred == 0}

def run_batched(policy, X0, oracle, eps_total, mode='ppo', rng=None, range_clip=False):
    """Run one episode per row of X0 in parallel (10 steps max). mode: 'ppo' (deterministic policy) or 'random'."""
    N = len(X0); x_cur = X0.copy(); last_applied = np.zeros((N, N_CYBER), dtype=np.float32)
    last_pred = np.concatenate([oracle.predict(X0[i:i + 65536]) for i in range(0, N, 65536)]).astype(np.int64); clean_pred = last_pred.copy()
    active = np.ones(N, bool); evaded = np.zeros(N, bool); steps = np.zeros(N, np.int64)
    if mode == 'ppo': h = np.zeros((1, N, 64), np.float32); c = np.zeros((1, N, 64), np.float32)
    for step in range(MAX_STEPS):
        idx = np.where(active)[0]
        if len(idx) == 0: break
        obs = build_obs(x_cur[idx], last_applied[idx], last_pred[idx], step)
        if mode == 'ppo':
            act = np.zeros((len(idx), N_CYBER), np.float32)
            for s in range(0, len(idx), EVAL_CHUNK):
                sl = idx[s:s + EVAL_CHUNK]
                a, (hn, cn) = policy.predict(obs[s:s + EVAL_CHUNK], state=(h[:, sl], c[:, sl]), episode_start=np.full(len(sl), step == 0), deterministic=True)
                act[s:s + EVAL_CHUNK] = a; h[:, sl] = hn; c[:, sl] = cn
        else:
            act = rng.uniform(-EPS_BOUND, EPS_BOUND, size=(len(idx), N_CYBER)).astype(np.float32)
        sub = x_cur[idx]; applied = apply_action(sub, X0[idx], act, EPS_BOUND, eps_total, range_clip); x_cur[idx] = sub; last_applied[idx] = applied
        pred = np.concatenate([oracle.predict(sub[i:i + 65536]) for i in range(0, len(sub), 65536)]).astype(np.int64); last_pred[idx] = pred
        ev = pred == 0; evaded[idx[ev]] = True; steps[idx[ev]] = step + 1; active[idx[ev]] = False
    return {'evaded': evaded, 'steps': steps, 'x_final': x_cur, 'pred_final': last_pred, 'clean_pred': clean_pred}

def per_class(vec_bool):
    return np.array([vec_bool[y_atk_test == c].mean() for c in range(1, 6)])
def mean_steps(res):
    return np.array([res['steps'][(y_atk_test == c) & res['evaded']].mean() if ((y_atk_test == c) & res['evaded']).any() else np.nan for c in range(1, 6)])
def summarise(res):
    disp = np.abs(res['x_final'][:, CYBER_IDX] - X_atk_test[:, CYBER_IDX])
    _ex = lambda Z: np.maximum(np.maximum(FEAT_LO[CYBER_IDX] - Z[:, CYBER_IDX], Z[:, CYBER_IDX] - FEAT_HI[CYBER_IDX]), 0).max(1)
    exc = np.maximum(_ex(res['x_final']) - _ex(X_atk_test), 0)    # sigma ADDED beyond what the clean test row already had outside the training min/max
    return {'per_class_evasion': per_class(res['evaded']), 'overall_evasion': float(res['evaded'].mean()), 'mean_steps_to_evade': mean_steps(res),
            'mean_queries': float(1 + np.where(res['evaded'], res['steps'], MAX_STEPS).mean()),
            'attack_f1': float(f1_score(y_atk_test, res['pred_final'], average='weighted')),
            'max_abs_disp_mean': float(disp.max(1).mean()), 'max_abs_disp_max': float(disp.max()), 'mean_excess_beyond_train_range': float(exc.mean()), 'frac_rows_beyond_range_0.5sigma': float((exc > 0.5).mean()),
            'evaded': res['evaded'].copy()}
print('environment + evaluator defined; cyber features controlled:', N_CYBER)

## 3. White-box attack (with per-round evasion trace)

In [ ]:
def get_thresholds(clf):
    """feature index -> sorted unique float32 split thresholds used anywhere in the ensemble."""
    df = clf.get_booster().trees_to_dataframe(); sp = df[df['Feature'] != 'Leaf']
    return {int(str(f).lstrip('f')): np.unique(g['Split'].to_numpy(dtype=np.float32)) for f, g in sp.groupby('Feature')}

class Margin:
    """Raw per-class scores of a fitted classifier (the white-box access). Scores in blocks and, if the GPU runs out of memory, halves the block and retries."""
    def __init__(self, clf): self.b = clf.get_booster(); self.bs = BLOCK
    def _pred(self, X):
        while True:
            try:
                return [np.asarray(self.b.inplace_predict(np.ascontiguousarray(X[s:s + self.bs], dtype=np.float32), predict_type='margin', validate_features=False)) for s in range(0, len(X), self.bs)]
            except xgb.core.XGBoostError as e:
                if 'out of memory' not in str(e).lower() or self.bs <= 20000: raise
                self.bs = max(20000, self.bs // 2); print(f'      GPU out of memory -> scoring in blocks of {self.bs:,} rows', flush=True)
    def __call__(self, X):
        out = self._pred(X); return np.concatenate(out) if len(out) > 1 else out[0]
def gap_of(z): return (z[:, 0] - z[:, 1:].max(1)).astype(np.float64)       # > 0  <=>  benign is the arg-max class

def interval(o, j, B):
    """allowed interval of feature j for rows with original values o: L-inf ball (if B) intersected with the training range (never further out than the row already is)."""
    lo_r, hi_r = np.minimum(FEAT_LO[j], o), np.maximum(FEAT_HI[j], o)
    if B is None: return lo_r.astype(np.float32), hi_r.astype(np.float32)
    return np.maximum(o - np.float32(B), lo_r).astype(np.float32), np.minimum(o + np.float32(B), hi_r).astype(np.float32)

def candidates(xc, X0c, j, T, B, K):
    """candidate (row, value) moves for feature j: the K nearest thresholds on each side inside the allowed interval, plus the interval ends."""
    xv = xc[:, j]; a, b = interval(X0c[:, j], j, B); nT = len(T); idx = np.searchsorted(T, xv, side='right'); R, V = [], []
    for d in range(min(K, nT)):
        k = idx + d; v = T[np.minimum(k, nT - 1)]; ok = (k < nT) & (v <= b); R.append(np.flatnonzero(ok)); V.append(v[ok])
        k = idx - 1 - d; v = np.nextafter(T[np.maximum(k, 0)], np.float32(-np.inf)); ok = (k >= 0) & (v >= a); R.append(np.flatnonzero(ok)); V.append(v[ok])
    ok = b > xv; R.append(np.flatnonzero(ok)); V.append(b[ok]); ok = a < xv; R.append(np.flatnonzero(ok)); V.append(a[ok])
    return np.concatenate(R), np.concatenate(V).astype(np.float32)

def one_step(margin, xc, X0c, g, feats, TH, B, K):
    """best single-feature move per row (by the model's own margin gap). Returns best_gap, best_feature (-1 = none helps), best_value, n_scored."""
    n = len(xc); best_g = g.copy(); best_j = np.full(n, -1, np.int64); best_v = np.zeros(n, np.float32); n_scored = 0
    bufR, bufJ, bufV = [], [], []; size = 0
    def flush():
        nonlocal n_scored
        if not bufR: return
        R, J, V = np.concatenate(bufR), np.concatenate(bufJ), np.concatenate(bufV); bufR.clear(); bufJ.clear(); bufV.clear()
        for s in range(0, len(R), BLOCK):
            r, jj, v = R[s:s + BLOCK], J[s:s + BLOCK], V[s:s + BLOCK]
            Xc = xc[r]; Xc[np.arange(len(r)), jj] = v                    # xc[r] is a copy
            gg = gap_of(margin(Xc)); n_scored += len(r)
            o = np.argsort(r, kind='stable'); rs, gs = r[o], gg[o]
            starts = np.flatnonzero(np.r_[True, rs[1:] != rs[:-1]]); segmax = np.maximum.reduceat(gs, starts); cnt = np.diff(np.r_[starts, len(rs)])
            hit = np.flatnonzero(gs == np.repeat(segmax, cnt)); _, first = np.unique(rs[hit], return_index=True); sel = o[hit[first]]
            rows = r[sel]; better = gg[sel] > best_g[rows]; rr = rows[better]
            best_g[rr] = gg[sel][better]; best_j[rr] = jj[sel][better]; best_v[rr] = v[sel][better]
    for j in feats:
        T = TH.get(int(j))
        if T is None: continue
        R, V = candidates(xc, X0c, int(j), T, B, K)
        if len(R) == 0: continue
        bufR.append(R); bufJ.append(np.full(len(R), j, np.int64)); bufV.append(V); size += len(R)
        if size >= BLOCK: flush(); size = 0
    flush(); return best_g, best_j, best_v, n_scored

def whitebox_trace(margin, X0, y0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):
    """as NB3c's whitebox, plus a trace: per round, the evaded share per class (classes 4, 5) and the median margin gap of the rows not yet evaded."""
    x = X0.copy(); g = gap_of(margin(x)); active = g <= 0; stalled = np.zeros(len(x), bool); trace = []; t_start = time.time()
    active = ~(g > 0); init_ev = g > 0
    for it in range(max_iters):
        act = np.flatnonzero(active)
        if len(act) == 0: break
        t0 = time.time(); scored = 0
        for s in range(0, len(act), chunk):
            ids = act[s:s + chunk]; bg, bj, bv, ns = one_step(margin, x[ids], X0[ids], g[ids], feats, TH, B, K); scored += ns
            imp = bj >= 0; ii = ids[imp]; x[ii, bj[imp]] = bv[imp]; g[ii] = bg[imp]; stalled[ids[~imp]] = True; active[ids[~imp]] = False
        active &= (g <= 0); ev = g > 0
        trace.append({'round': it + 1, 'active': int(len(act)), 'evaded_all': float(ev.mean()), 'evaded_pc': [float(ev[y0 == c].mean()) for c in range(1, 6)], 'evaded_c4': float(ev[y0 == 4].mean()), 'evaded_c5': float(ev[y0 == 5].mean()),
                      'median_gap_unevaded': float(np.median(g[~ev])) if (~ev).any() else float('nan'), 'seconds': time.time() - t0})
        if verbose and ((it + 1) % 10 == 0 or it == 0): print(f'      round {it + 1:>3}: active {len(act):>5} | evaded ransomware {trace[-1]["evaded_c4"]:.3f} reverseshell {trace[-1]["evaded_c5"]:.3f} | median gap of unevaded {trace[-1]["median_gap_unevaded"]:+.3f} | {time.time() - t0:4.1f}s', flush=True)
    return x, g, init_ev, stalled, trace
def whitebox(margin, X0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):      # used by the self-test
    return whitebox_trace(margin, X0, np.zeros(len(X0), dtype=np.int64), feats, TH, B, K, max_iters, chunk, verbose)
print('white-box attack defined')

In [ ]:
def selftest(clf, margin, TH, seed=0):
    rng = np.random.RandomState(seed); rows = rng.choice(len(X_atk_test), 60, replace=False); X0 = X_atk_test[rows]; feats = rng.choice(CYBER_IDX, 12, replace=False)
    g0 = gap_of(margin(X0)); worst = 0.0
    for B in [0.1, 0.5, None]:
        bg, bj, bv, _ = one_step(margin, X0.copy(), X0, g0, feats, TH, B, K=10 ** 6)
        for i in range(len(rows)):
            best = g0[i]
            for j in feats:
                T = TH.get(int(j))
                if T is None: continue
                a, b = interval(X0[i:i + 1, j], int(j), B); a, b = float(a[0]), float(b[0]); inner = T[(T > a) & (T < b)]
                pts = np.concatenate([[a, b], (inner[:-1] + inner[1:]) / 2 if len(inner) > 1 else [], inner, np.nextafter(inner, np.float32(-np.inf))]).astype(np.float32)
                pts = pts[(pts >= a) & (pts <= b)]
                Xc = np.repeat(X0[i:i + 1], len(pts), 0); Xc[:, j] = pts; best = max(best, gap_of(margin(Xc)).max())
            worst = max(worst, best - bg[i])
            assert bg[i] >= best - 1e-4, f'one_step missed a better move (row {i}, budget {B}): {bg[i]} < {best}'
    # constraints after a short attack
    sub = X_atk_test[rng.choice(len(X_atk_test), 400, replace=False)]
    for B in [0.1, 0.5]:
        xa, ga, ie, st, lg = whitebox(margin, sub, CYBER_IDX, TH, B, max_iters=4, verbose=False)
        d = np.abs(xa - sub); assert d[:, CYBER_IDX].max() <= B + 1e-6, 'L-inf budget violated'
        other = np.setdiff1d(np.arange(N_FEAT), CYBER_IDX); assert d[:, other].max() == 0, 'a non-controllable feature changed'
        lo_ok = (xa >= np.minimum(FEAT_LO, sub) - 1e-6) & (xa <= np.maximum(FEAT_HI, sub) + 1e-6); assert lo_ok.all(), 'left the training range'
        assert (ga >= gap_of(margin(sub)) - 1e-9).all(), 'margin gap decreased'
    print(f'self-test passed (largest shortfall of one_step vs full enumeration: {max(worst, 0):.2e}); budget, range, non-controllable features and monotone margin all hold')

## 4. The same 10,000 test rows as NB5 / NB5b

In [ ]:
rng0 = np.random.RandomState(0); pick = np.sort(np.concatenate([rng0.choice(np.flatnonzero(y_atk_test == c), N_WB_PER_CLASS, replace=False) for c in range(1, 6)]))
X_wb, y_wb = X_atk_test[pick], y_atk_test[pick]
def per_class_wb(ev): return np.array([ev[y_wb == c].mean() for c in range(1, 6)])
print(f'white-box rows: {len(X_wb)} ({N_WB_PER_CLASS} per attack class) | budgets {WB_BUDGETS} | up to {MAX_ITERS} rounds')

## 5. Per-seed run

In [ ]:
def load_model(path, seed):
    gpu = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); gpu.load_model(path)
    cpu = xgb.XGBClassifier(); cpu.load_model(path); cpu.set_params(device='cpu', n_jobs=-1); return gpu, cpu
R = {}; t_all = time.time()
for seed in SEEDS:
    pf = f'defence1c_seed{seed}_{PIPELINE_VERSION}.pkl'; cfg_fp = repr((LONG_ITERS, CHECKPOINTS, LONG_TAGS, K_CAND, WB_BUDGETS, N_WB_PER_CLASS, seed, X_atk_test.shape, float(X_atk_test[:5].sum()), 'nb5c-v1'))
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: R[seed] = saved['R']; print(f'seed {seed}: result file found -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}', flush=True)
    mpath = f'xgb_robust_{PIPELINE_VERSION}_seed{seed}.json'; assert os.path.exists(mpath), f'{mpath} missing (NB5b)'
    b = joblib.load(f'defence1b_seed{seed}_{PIPELINE_VERSION}.pkl')['R']; d = joblib.load(f'defence1_seed{seed}_{PIPELINE_VERSION}.pkl')['R']
    gm, cm = load_model(mpath, seed); R[seed] = {'clean_f1_reload': float(f1_score(y_test, cm.predict(X_test), average='weighted'))}
    print(f'  reloaded robust model clean F1 {R[seed]["clean_f1_reload"]:.4f}', flush=True)
    mg = Margin(gm); TH = get_thresholds(gm)
    for tag in LONG_TAGS:
        ts = time.time(); xa, ga, ie, st, tr = whitebox_trace(mg, X_wb, y_wb, CYBER_IDX, TH, WB_BUDGETS[tag], max_iters=LONG_ITERS, verbose=False)
        ev = cm.predict(xa) == 0; pc = per_class_wb(ev)
        curve = {c: (tr[min(c, len(tr)) - 1]['evaded_all'] if len(tr) else float(ie.mean())) for c in CHECKPOINTS}      # search-side (GPU margin) evasion after c rounds
        pcc = {c: (np.array(tr[min(c, len(tr)) - 1]['evaded_pc']) if len(tr) else np.zeros(5)) for c in CHECKPOINTS}
        R[seed][tag] = {'per_class_evasion': pc, 'overall_evasion': float(ev.mean()), 'curve': curve, 'curve_pc': pcc, 'rounds_run': len(tr), 'still_improving': float((~st & ~ev).mean()), 'stalled': float((st & ~ev).mean()), 'search_vs_cpu_disagree': float(((ga > 0) != ev).mean()),
                        'nb5b_evasion_60': float(b[f'wb_{tag}|rob']['overall_evasion']), 'nb5b_still_improving_60': float(b[f'wb_{tag}|rob']['rows_round_limit'])}
        print(f'  {tag} sigma: final evasion {ev.mean():.3f} (NB5b @60 rounds {b[f"wb_{tag}|rob"]["overall_evasion"]:.3f}, NB5 defended {d[f"wb_{tag}|def"]["overall_evasion"]:.3f}) | curve ' + ' '.join(f'{c}:{v:.3f}' for c, v in curve.items()) + f' | per-class {np.round(pc, 3).tolist()} | still improving {R[seed][tag]["still_improving"]:.3f} stalled {R[seed][tag]["stalled"]:.3f} | rounds run {len(tr)} | {(time.time() - ts) / 60:.1f} min', flush=True)
    R[seed]['nb5'] = {t: d[f'wb_{t}|def'] for t in LONG_TAGS}; R[seed]['nb5b60'] = {t: b[f'wb_{t}|rob'] for t in LONG_TAGS}
    joblib.dump({'fp': cfg_fp, 'R': R[seed]}, pf); print(f'  seed {seed} done in {(time.time() - t0) / 60:.1f} min', flush=True); del mg, gm, cm; torch.cuda.empty_cache()
print(f'Total {(time.time() - t_all) / 60:.1f} min')

## 6. Results

In [ ]:
atk_names = class_names[1:]
def ptest(a, b):
    d = a - b
    if len(d) < 2: return float('nan')
    return 1.0 if d.std(ddof=1) == 0 and d.mean() == 0 else (0.0 if d.std(ddof=1) == 0 else float(stats.ttest_rel(a, b).pvalue))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rk, i in enumerate(o): run = max(run, (len(p) - rk) * p[i]); adj[i] = min(1.0, run)
    return adj
ms = lambda x: f'{x.mean():.4f} +/- {(x.std(ddof=1) if len(x) > 1 else 0):.4f}'
print('6a. Reproduction check: robust-model clean F1 after reload vs NB5b'); print('   ', {s: round(R[s]['clean_f1_reload'], 4) for s in SEEDS})
rows = []
for t in LONG_TAGS:
    cur = {c: np.array([R[s][t]['curve'][c] for s in SEEDS]) for c in CHECKPOINTS}; fin = np.array([R[s][t]['overall_evasion'] for s in SEEDS]); nb5 = np.array([R[s]['nb5'][t]['overall_evasion'] for s in SEEDS]); b60 = np.array([R[s][t]['nb5b_evasion_60'] for s in SEEDS])
    rows.append({'budget': f'{t} sigma', 'NB5b @60': ms(b60), **{f'@{c}': f'{cur[c].mean():.3f}' for c in CHECKPOINTS[1:]}, 'final (CPU)': ms(fin), 'still improving': f'{np.mean([R[s][t]["still_improving"] for s in SEEDS]):.3f}', 'NB5 defended': ms(nb5), 'final-NB5': f'{(fin - nb5).mean():+.4f}', 'p': f'{ptest(fin, nb5):.4f}'})
conv_table = pd.DataFrame(rows); pd.set_option('display.width', 330); print('\n6b. Overall evasion vs attack rounds (curve = search-side, final = CPU decision)'); print(conv_table.to_string(index=False))
pcr = []
for t in LONG_TAGS:
    a_b = np.stack([R[s]['nb5b60'][t]['per_class_evasion'] for s in SEEDS]); a_f = np.stack([R[s][t]['per_class_evasion'] for s in SEEDS]); a_d = np.stack([R[s]['nb5'][t]['per_class_evasion'] for s in SEEDS]); adj = holm(np.nan_to_num([ptest(a_f[:, i], a_d[:, i]) for i in range(5)], nan=1.0))
    pcr.append({'budget': f'{t} sigma', **{c: f'{a_d[:, i].mean():.3f} -> {a_b[:, i].mean():.3f} -> {a_f[:, i].mean():.3f}' + ('*' if adj[i] < 0.05 else '') for i, c in enumerate(atk_names)}})
pc_table = pd.DataFrame(pcr); print('\n6c. Per-class evasion: NB5 defended -> robust @60 rounds -> robust @300 rounds (* = final differs from NB5 defended, Holm p < 0.05)'); print(pc_table.to_string(index=False))
print('\n6d. Has the curve flattened? (evasion change over the last 100 rounds, search side, mean over seeds)')
for t in LONG_TAGS:
    d_ = np.mean([R[s][t]['curve'][300] - R[s][t]['curve'][200] for s in SEEDS]); d2 = np.mean([R[s][t]['curve'][200] - R[s][t]['curve'][100] for s in SEEDS]); print(f'   {t} sigma: 100->200 {d2:+.4f}, 200->300 {d_:+.4f}  ->  {"flat (converged)" if abs(d_) < 0.005 and R[SEEDS[0]][t]["still_improving"] < 0.01 else "still moving / rows still improving"}')
print('\n6e. Search/CPU disagreement (should be ~0):', {t: round(float(np.mean([R[s][t]["search_vs_cpu_disagree"] for s in SEEDS])), 5) for t in LONG_TAGS})

## 7. Figure and save

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
for t in LONG_TAGS:
    v = np.array([[R[s][t]['curve'][c] for c in CHECKPOINTS] for s in SEEDS]); ax[0].errorbar(CHECKPOINTS, v.mean(0), v.std(0, ddof=1) if len(SEEDS) > 1 else None, marker='o', capsize=3, label=f'robust, {t} sigma')
    ax[0].axhline(np.mean([R[s]['nb5'][t]['overall_evasion'] for s in SEEDS]), ls=':', color='grey', lw=.8)
ax[0].set_xlabel('white-box rounds'); ax[0].set_ylabel('overall evasion'); ax[0].set_title('Robust models: evasion vs attack rounds (dotted = NB5 defended)'); ax[0].legend(); ax[0].grid(alpha=.3)
for c, cn in enumerate(atk_names): ax[1].plot([float(t) for t in LONG_TAGS], [np.mean([R[s][t]['per_class_evasion'][c] for s in SEEDS]) for t in LONG_TAGS], marker='o', label=cn)
ax[1].set_xlabel('budget (sigma)'); ax[1].set_ylabel('evasion after 300 rounds'); ax[1].set_title('Per class'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'defence1c_convergence_{PIPELINE_VERSION}.png', dpi=150); plt.show()
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'conv_table': conv_table, 'per_class_table': pc_table, 'per_seed': {s: R[s] for s in SEEDS}, 'config': {'LONG_ITERS': LONG_ITERS, 'CHECKPOINTS': CHECKPOINTS, 'LONG_TAGS': LONG_TAGS}}, f'defence1c_results_{PIPELINE_VERSION}.pkl')
conv_table.to_csv(f'defence1c_convergence_table_{PIPELINE_VERSION}.csv', index=False); pc_table.to_csv(f'defence1c_per_class_table_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB5c', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'xgboost': xgb.__version__, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb5c_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'defence1c_results_{PIPELINE_VERSION}.pkl', f'defence1c_convergence_table_{PIPELINE_VERSION}.csv', f'defence1c_per_class_table_{PIPELINE_VERSION}.csv', f'defence1c_convergence_{PIPELINE_VERSION}.png', f'nb5c_run_manifest_{PIPELINE_VERSION}.json']: print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)